In [ ]:
import os, sys
from pathlib import Path
# Walk up from cwd to find repo ROOT (dir containing src/data_cmapss.py).
# Works whether kernel cwd is repo root, notebooks/, or elsewhere.
_here = Path().resolve()
ROOT = next((p for p in [_here, *_here.parents] if (p / 'src' / 'data_cmapss.py').exists()), None)
if ROOT is None:
    raise ModuleNotFoundError(f"Could not find repo ROOT with src/data_cmapss.py from CWD={_here}")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('ROOT:', ROOT)

# 02 — FD001 dataset analysis
Calls `src/data_cmapss.py`. Findings feed `docs/DATASET.md`.

In [ ]:
import os, sys
from pathlib import Path
# Bootstrap (same as cell 1) so this cell also works with 'Run cell' alone.
_here = Path().resolve()
_root = next((p for p in [_here, *_here.parents] if (p / 'src' / 'data_cmapss.py').exists()), None)
if _root is None:
    raise ModuleNotFoundError(f"Could not find repo ROOT with src/data_cmapss.py from CWD={_here}. Run cell 1 first.")
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from src.data_cmapss import load_fd001, add_rul, SENSOR_COLS
tr, te, rul = load_fd001('CMAPSSData')
print('train', tr.shape, '| test', te.shape, '| rul', rul.shape)
print('train engines', tr['unit'].nunique(), '| test engines', te['unit'].nunique())
print(tr.groupby('unit')['cycle'].max().describe().round(1))
print('test RUL:', rul.describe().round(1).to_dict())
print('sensor std (constant ~0):')
print(tr[SENSOR_COLS].std().round(4).to_string())

In [ ]:
# Deeper analysis: RUL distribution on train (capped at 125)
from src.data_cmapss import add_capped_rul
tr = add_capped_rul(add_rul(tr), cap=125)
print('Train RUL (capped 125):', tr['rul_capped'].describe().round(1).to_dict())
print('Train RUL (uncapped):', tr['rul'].describe().round(1).to_dict())
print('Max cycle per engine:', tr.groupby('unit')['cycle'].max().describe().round(1))

In [ ]:
# Check constant sensors (should have near-zero std)
sensor_std = tr[SENSOR_COLS].std().round(6)
constant = sensor_std[sensor_std < 0.01].index.tolist()
print('Near-constant sensors:', constant)
print('Informative sensors:', [s for s in SENSOR_COLS if s not in constant])

In [ ]:
# Engine-level split verification
from src.data_cmapss import engine_split, assert_disjoint
units = sorted(int(u) for u in tr['unit'].unique())
tr_units, va_units = engine_split(units, 0.2, 42)
print(f'Train engines: {len(tr_units)}, Val engines: {len(va_units)}')
print('Train units:', tr_units[:10], '...')
print('Val units:', va_units)
assert_disjoint(tr_units, va_units)
print('✓ Train/Val engine split is disjoint')

In [ ]:
# Test set: verify unit IDs are 1-100 but distinct fleet from train
test_units = sorted(int(u) for u in te['unit'].unique())
print('Test units:', test_units[:10], '...')
print('Test engines:', len(test_units))
print('RUL file length:', len(rul))
print('RUL matches test engines:', len(rul) == len(test_units))

## Summary for `docs/DATASET.md`
- **Source**: NASA PCoE CMAPSS Jet Engine Simulated Data
- **FD001 scope**: 100 train + 100 test engines; single operating condition; single fault mode (HPC degradation)
- **Train**: 20,631 rows, max cycle 362, mean RUL 107.8 (capped-125 mean 86.8)
- **Test**: 13,096 rows, 100 engines, true RUL mean 75.5 (min 7, max 145)
- **Constant sensors** (std≈0): s1, s5, s6, s10, s16, s18, s19 → exclude from features
- **Leakage policy**: Split by ENGINE (unit), never by row; test fleet IDs overlap train but are distinct
- **Target**: Remaining Useful Life (RUL) in cycles; piecewise-linear capped RUL (cap=125) for training